# FailureSensorIQ GEPA Optimization

In [17]:
# Library Imports
import os
from datasets import load_dataset
import dspy
from dspy import GEPA

import json
import random
import re
import textwrap
from typing import Optional

In [18]:
# Global Variables
seed = 42
n_train_assets = 2


In [19]:
# Watsonx AI Credentials
from ibm_watsonx_ai import APIClient,Credentials
api_key = os.getenv("WATSONX_API_KEY")
url = "https://us-south.ml.cloud.ibm.com"
project_id = os.getenv("WATSONX_PROJECT_ID")

#Testing Credentials 
credentials = Credentials(api_key=api_key, url=url)

client = APIClient(credentials)

client.set.default_project(project_id)

'SUCCESS'

In [20]:
# Setup DSPy

import dspy
model_name = "watsonx/openai/gpt-oss-120b"
max_tokens = 32000
temperature = 0.7

lm = dspy.LM(model=model_name, max_tokens=max_tokens, api_key=api_key,temperature = temperature
             ,project_id = project_id
             ,api_base=url
             )

dspy.configure(lm=lm, temperature=1, trace=[], experimental=True)

In [21]:
# Helper Functions

def _parse_answer(option_ids: list[str], correct: list[bool]) -> str:
    """
    Convert the dataset's boolean correct list into a letter string.

    Example:
        option_ids = ["A", "B", "C", "D", "E"]
        correct    = [False, False, False, True, False]
        → "D"

        option_ids = ["A", "B", "C"]
        correct    = [True, False, True]
        → "A,C"
    """
    letters = [lid.upper() for lid, flag in zip(option_ids, correct) if flag]
    return ",".join(sorted(letters))


def _build_options_str(options: list[str], option_ids: list[str]) -> str:
    """
    Render options as a lettered multi-line string.

    Example:
        options    = ["partial discharge", "resistance", "current"]
        option_ids = ["A", "B", "C"]
        →  "  A) partial discharge\n  B) resistance\n  C) current"
    """
    return "\n".join(
        f"  {lid}) {text}"
        for lid, text in zip(option_ids, options)
    )


def _collect_and_shuffle(assets: list[str], data_dict: dict[str, list[dspy.Example]]) -> list[dspy.Example]:

    rng = random.Random(seed)
    combined = [ex for a in assets for ex in data_dict.get(a, [])]
    rng.shuffle(combined)
    return combined

## Load FailureSensorIQ Dataset

In [22]:
 # Load FailureSensorIQ Dataset

ds = load_dataset("ibm-research/FailureSensorIQ", "single_true_multi_choice_qa")


In [23]:

def init_dataset_splits(ds):

    fsiq_dict: dict[str, list[dspy.Example]] = {}

    # Assets used for Training
    n_train_assets = 2

    # Parsing Rows into dspy.Examples
    for row in ds["train"]:

        q_type = row.get("question_type", "")

        options_str = _build_options_str(row["options"], row["option_ids"])
        answer      = _parse_answer(row["option_ids"], row["correct"])
        asset       = row.get("asset_name", "unknown")

        ex = dspy.Example(
            question      = row["question"],
            options       = options_str,
            answer        = answer,
            asset         = asset,
            query_type    = row.get("relevancy", "unknown"),
            question_type = q_type,
        ).with_inputs("question", "options")
    
        fsiq_dict.setdefault(asset, []).append(ex)

    train_assets     = [asset for i, (asset, examples) in enumerate(fsiq_dict.items()) if i < n_train_assets]
    remaining_assets = [asset for i, (asset, examples) in enumerate(fsiq_dict.items()) if i >= n_train_assets]


    val_assets  = [a for i, a in enumerate(remaining_assets) if i % 2 == 0]
    test_assets = [a for i, a in enumerate(remaining_assets) if i % 2 == 1]

    print("train_assets :" , train_assets)
    print("val_assets   :" , val_assets)
    print("test_assets  :" , test_assets)

    train_set = _collect_and_shuffle(train_assets, fsiq_dict)
    val_set   = _collect_and_shuffle(val_assets, fsiq_dict)
    test_set  = _collect_and_shuffle(test_assets, fsiq_dict) #*5  

    print(f"train_set: {len(train_set)} examples")
    print(f"val_set: {len(val_set)} examples")
    print(f"test_set: {len(test_set)} examples")

    return train_set, val_set, test_set

In [24]:
train_set, val_set, test_set = init_dataset_splits(ds)

train_assets : ['electric motor', 'steam turbine']
val_assets   : ['aero gas turbine', 'pump', 'reciprocating internal combustion engine', 'fan']
test_assets  : ['industrial gas turbine', 'compressor', 'electric generator', 'power transformer']
train_set: 405 examples
val_set: 1024 examples
test_set: 1238 examples


## `dspy.chainofthought` Setup

In [25]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    question  : str = dspy.InputField(desc="The MCQA question about failure modes or sensors.")
    options   : str = dspy.InputField(desc="Lettered answer choices, one per line.")
    reasoning : str = dspy.OutputField(desc="Step-by-step reasoning referencing FMEA knowledge.")
    answer    : str = dspy.OutputField(desc="Correct letter(s), e.g. 'D' or 'A,C'.")

program = dspy.ChainOfThought(GenerateResponse)



### Defining The Evaluation Metric

In [26]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """
    Metric that checks if prediction is one of the valid options (A-E or combinations like A,C)
    and matches the correct answer.
    """
    correct_answer = example['answer']
    
    try:
        llm_answer = str(prediction.answer).strip().upper()
        
        # Validate that answer is one of the valid options (single letter or comma-separated letters A-E)
        # Valid formats: "A", "B", "A,C", "B,D", etc.
        answer_pattern = r'^[A-E](,[A-E])*$'
        
        if not re.match(answer_pattern, llm_answer):
            return 0
        
        # Ensure answers are sorted (e.g., "A,C" not "C,A") to match expected format
        parts = llm_answer.split(',')
        llm_answer_normalized = ','.join(sorted(set(parts)))
        correct_answer_normalized = ','.join(sorted(set(correct_answer.split(','))))
        
    except (ValueError, AttributeError, TypeError):
        return 0
    
    return int(correct_answer_normalized == llm_answer_normalized)

### Evaluating the Unoptimized Baseline Chain of Thought

In [27]:
evaluate = dspy.Evaluate(
    devset=train_set, #change to test_set for final evaluation
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 247.00 / 405 (61.0%): 100%|██████████| 405/405 [00:01<00:00, 227.73it/s]

2026/03/06 19:25:24 INFO dspy.evaluate.evaluate: Average Metric: 247 / 405 (61.0%)


,question,options,example_answer,asset,query_type,question_type,reasoning,pred_answer,metric
0,"When a electric motor has unbalance, which sensor out of the choic...",A) vibration\n B) resistance,B,electric motor,irrelevant_sensors_for_failure_mode,mcp1_negative,Unbalance in an electric motor is a mechanical condition that crea...,B,✔️ [1]
1,"In electric motor, which failure event is unimportant if the senso...",A) loss of input power phase\n B) rotor windings fault\n C) st...,A,electric motor,irrelevant_failure_modes_for_sensor,mcp1_negative,"In an FMEA for an electric motor, the sensor that monitors the coo...",A,✔️ [1]
2,"In electric motor, which failure event is unimportant if the senso...",A) brush(es) fault\n B) misalignment\n C) rotor windings fault...,B,electric motor,irrelevant_failure_modes_for_sensor,mcp1_negative,The torque sensor monitors the motor’s output torque. - **Brush(es...,D,✔️ [0]
3,What is the most relevant failure mode for steam turbine if coast ...,A) misalignment\n B) damaged labyrinth\n C) unequal expansion\...,B,steam turbine,relevant_failure_modes_for_sensor,mcp1_positive,Coast‑down time is a diagnostic test that measures how quickly a t...,B,✔️ [1]
4,"For steam turbine, which failure mode is pertinent if coast down t...",A) misalignment\n B) unequal expansion\n C) bearing wear\n D)...,C,steam turbine,relevant_failure_modes_for_sensor,mcp1_positive,Coast‑down testing measures how quickly the turbine rotor decelera...,C,✔️ [1]
...,...,...,...,...,...,...,...,...,...
400,Which sensor out of the choices is not effective in indicating the...,A) vibration\n B) axial flux,B,electric motor,irrelevant_sensors_for_failure_mode,mcp1_negative,Unbalance in a rotating electric motor creates a cyclic centrifuga...,B,✔️ [1]
401,What is the irrelevant failure event for steam turbine if the sens...,A) eccentric rotor\n B) damaged labyrinth,A,steam turbine,irrelevant_failure_modes_for_sensor,mcp1_negative,The speed sensor of a steam turbine monitors the rotational speed ...,B,✔️ [0]
402,Which sensor out of the choices can indicate the presence of stato...,A) resistance\n B) partial discharge\n C) power\n D) cooling ...,D,electric motor,relevant_sensors_for_failure_mode,mcp1_positive,"In an FMEA for an electric motor, a **stator winding fault** (shor...",B,✔️ [0]
403,"If vibration in electric motor shows abnormal readings, which fail...",A) stator windings fault\n B) insulation deterioration\n C) br...,A,electric motor,relevant_failure_modes_for_sensor,mcp1_positive,"In an FMEA of an electric motor, the *vibration* symptom is primar...",C,✔️ [0]


EvaluationResult(score=60.99, results=<list of 405 results>)

## GEPA Optimization

In [28]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """
    Metric that checks if prediction is one of the valid options (A-E or combinations like A,C)
    and matches the correct answer.
    """
    correct_answer = example['answer']
    
    try:
        llm_answer = str(prediction.answer).strip().upper()
        
        # Validate that answer is one of the valid options (single letter or comma-separated letters A-E)
        # Valid formats: "A", "B", "A,C", "B,D", etc.
        answer_pattern = r'^[A-E](,[A-E])*$'
        
        if not re.match(answer_pattern, llm_answer):
            return 0
        
        # Ensure answers are sorted (e.g., "A,C" not "C,A") to match expected format
        parts = llm_answer.split(',')
        llm_answer_normalized = ','.join(sorted(set(parts)))
        correct_answer_normalized = ','.join(sorted(set(correct_answer.split(','))))
        
    except (ValueError, AttributeError, TypeError):
        feedback_text = f"The final answer must be one of the options (A-E or combinations like A,C). You responded with '{prediction.answer}', which couldn't be parsed as a valid option. Please ensure your answer is a valid option without any additional text or formatting."
        feedback_text += f" The correct answer is '{correct_answer}'."      


        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)

    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."

    return int(correct_answer_normalized == llm_answer_normalized)

In [29]:
optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=dspy.LM(model=model_name, max_tokens=max_tokens, api_key=api_key,tenperature = temperature
             ,project_id = project_id   
             ,api_base=url)
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/03/06 19:25:24 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 4476 metric calls of the program. This amounts to 3.13 full evals on the train+val set.
2026/03/06 19:25:24 INFO dspy.teleprompt.gepa.gepa: Using 1024 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/4476 [00:00<?, ?rollouts/s]2026/03/06 19:25:27 INFO dspy.evaluate.evaluate: Average Metric: 646 / 1024 (63.1%)
2026/03/06 19:25:27 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.630859375 over 1024 / 1024 examples
GEPA Optimization:  23%|██▎       | 1024/4476 [00:03<00:11, 292.49rollouts/s]2026/03/06 19:25:27 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 338.34it/s]

2026/03/06 19:25:27 INFO dspy.evaluate.evaluate: Average Metric: 3 / 3 (100.0%)
2026/03/06 19:25:27 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/03/06 19:25:27 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
2026/03/06 19:25:27 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.630859375



Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 295.01it/s]

2026/03/06 19:25:27 INFO dspy.evaluate.evaluate: Average Metric: 0 / 3 (0.0%)
2026/03/06 19:25:27 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Proposed new text for predict: You will be given a short multiple‑choice problem.  
The input consists of:

* A **question** describing a situation or asking for the best answer.  
* A list of **options** labeled with single uppercase letters (A, B, C, …).  
  Each option is on its own line and may contain additional text after the letter.

Your task is to determine which option correctly answers the question, using the factual knowledge demonstrated in the examples:

* For electric‑motor problems, an abnormal temperature reading points to **insulation deterioration** (option B in the first example).
* When asked which sensor does **not** significantly help detect bearing wear in a steam turbine, the correct answer is the **steam‑leakage sensor** (option D in the second example).
* When asked which sensor is **least useful** for detecting loss o

...


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 429.11it/s]

2026/03/06 19:25:41 INFO dspy.evaluate.evaluate: Average Metric: 3 / 3 (100.0%)
2026/03/06 19:25:41 INFO dspy.teleprompt.gepa.gepa: Iteration 45: All subsample scores perfect. Skipping.
2026/03/06 19:25:41 INFO dspy.teleprompt.gepa.gepa: Iteration 45: Reflective mutation did not propose a new candidate
2026/03/06 19:25:41 INFO dspy.teleprompt.gepa.gepa: Iteration 46: Selected program 1 score: 0.6572265625



Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00, 282.04it/s]

2026/03/06 19:25:41 INFO dspy.evaluate.evaluate: Average Metric: 1 / 3 (33.3%)
2026/03/06 19:25:41 INFO dspy.teleprompt.gepa.gepa: Iteration 46: Proposed new text for predict: **Task Overview**

You will be given a single multiple‑choice question that belongs to the domain of electric‑motor fault diagnosis.  
Each input consists of two parts:

1. **question** – a prompt that asks which *failure event* or *sensor* is **not relevant / should be excluded / is least likely** for a described symptom or condition.  
2. **options** – a list of answer choices, each prefixed with an upper‑case letter (A, B, C, …).

Your job is to:

* Use **electric‑motor knowledge** to decide which option correctly answers the question.
* Write a concise logical *reasoning* (2‑4 sentences) that explains why the chosen option is correct **and** why the other options are not appropriate.
* End with a single line `### answer` followed by the **letter** of the correct choice (e.g., `A`).

No extra text may appear b

2026/03/06 19:25:41 INFO dspy.teleprompt.gepa.gepa: Iteration 46: New subsample score 2 is better than old score 1. Continue to full eval and add to candidate pool.
2026/03/06 19:25:45 INFO dspy.evaluate.evaluate: Average Metric: 673 / 1024 (65.7%)
2026/03/06 19:25:45 INFO dspy.teleprompt.gepa.gepa: Iteration 46: Valset score for new program: 0.6572265625 (coverage 1024 / 1024)
2026/03/06 19:25:45 INFO dspy.teleprompt.gepa.gepa: Iteration 46: Val aggregate for new program: 0.6572265625
2026/03/06 19:25:45 INFO dspy.teleprompt.gepa.gepa: Iteration 46: Individual valset scores for new program: {0: 1, 1: 0, 2: 1, 3: 1, 4: 0, 5: 0, 6: 0, 7: 1, 8: 0, 9: 0, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 1, 18: 1, 19: 1, 20: 1, 21: 1, 22: 1, 23: 0, 24: 1, 25: 1, 26: 1, 27: 1, 28: 0, 29: 1, 30: 1, 31: 1, 32: 0, 33: 1, 34: 0, 35: 0, 36: 1, 37: 1, 38: 1, 39: 1, 40: 0, 41: 1, 42: 1, 43: 0, 44: 0, 45: 1, 46: 1, 47: 1, 48: 1, 49: 0, 50: 0, 51: 1, 52: 1, 53: 1, 54: 0, 55: 0, 56: 1, 57: 0, 58: 

# Checking the generated prompt

In [30]:
print(optimized_program.predict.signature.instructions)

**Task Overview**

You will be given a multiple‑choice question that relates to electric‑motor fault diagnosis.  
Each input contains:

1. **Question** – asks which failure event or sensor is *not relevant*, *should be excluded*, or *is least likely* for a specific symptom or fault condition.  
2. **Options** – a list of answer choices labeled with capital letters (A, B, C, …).  

Your job is to:

* Determine, using **electric‑motor knowledge**, which option is the correct answer.
* Write a brief, logical *reasoning* explaining why the selected option is the right one and why the other options are not appropriate.
* End with a single line `### answer` followed by the **letter** of the correct choice (e.g., `A`).

---

**Domain Knowledge You Must Apply**

1. **Abnormal speed‑sensor reading**
   * The speed sensor measures shaft rotational speed.
   * Faults that **directly affect speed** (electrical or control faults that change torque or cause the motor to stall) are *relevant*.
   * F

Evaluating the Optimized Program 

In [31]:
evaluate(optimized_program)

Average Metric: 265.00 / 405 (65.4%): 100%|██████████| 405/405 [00:05<00:00, 73.22it/s] 

2026/03/06 19:25:51 INFO dspy.evaluate.evaluate: Average Metric: 265 / 405 (65.4%)


,question,options,example_answer,asset,query_type,question_type,reasoning,pred_answer,metric
0,"When a electric motor has unbalance, which sensor out of the choic...",A) vibration\n B) resistance,B,electric motor,irrelevant_sensors_for_failure_mode,mcp1_negative,Unbalance in an electric motor creates a periodic radial force tha...,B,✔️ [1]
1,"In electric motor, which failure event is unimportant if the senso...",A) loss of input power phase\n B) rotor windings fault\n C) st...,A,electric motor,irrelevant_failure_modes_for_sensor,mcp1_negative,An abnormal reading from the cooling‑gas sensor indicates that the...,A,✔️ [1]
2,"In electric motor, which failure event is unimportant if the senso...",A) brush(es) fault\n B) misalignment\n C) rotor windings fault...,B,electric motor,irrelevant_failure_modes_for_sensor,mcp1_negative,Torque sensors detect changes in electromagnetic torque and the me...,B,✔️ [1]
3,What is the most relevant failure mode for steam turbine if coast ...,A) misalignment\n B) damaged labyrinth\n C) unequal expansion\...,B,steam turbine,relevant_failure_modes_for_sensor,mcp1_positive,Coast‑down time reflects the amount of drag that opposes the rotor...,B,✔️ [1]
4,"For steam turbine, which failure mode is pertinent if coast down t...",A) misalignment\n B) unequal expansion\n C) bearing wear\n D)...,C,steam turbine,relevant_failure_modes_for_sensor,mcp1_positive,Coast‑down time reflects how quickly the turbine rotor loses kinet...,C,✔️ [1]
...,...,...,...,...,...,...,...,...,...
400,Which sensor out of the choices is not effective in indicating the...,A) vibration\n B) axial flux,B,electric motor,irrelevant_sensors_for_failure_mode,mcp1_negative,Unbalance in an electric motor creates radial forces that generate...,B,✔️ [1]
401,What is the irrelevant failure event for steam turbine if the sens...,A) eccentric rotor\n B) damaged labyrinth,A,steam turbine,irrelevant_failure_modes_for_sensor,mcp1_negative,An abnormal speed‑sensor reading indicates a fault that changes th...,B,✔️ [0]
402,Which sensor out of the choices can indicate the presence of stato...,A) resistance\n B) partial discharge\n C) power\n D) cooling ...,D,electric motor,relevant_sensors_for_failure_mode,mcp1_positive,"A stator‑windings fault (e.g., shorted or open turns) directly alt...",A,✔️ [0]
403,"If vibration in electric motor shows abnormal readings, which fail...",A) stator windings fault\n B) insulation deterioration\n C) br...,A,electric motor,relevant_failure_modes_for_sensor,mcp1_positive,Abnormal vibration is primarily caused by forces that vary with th...,A,✔️ [1]


EvaluationResult(score=65.43, results=<list of 405 results>)